<a href="https://colab.research.google.com/github/noamzimer/my-site/blob/main/Matala4%2BFLOW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information Retrieval Evaluation Flow

This pipeline implements an end-to-end evaluation flow for search queries. The architecture is divided into three distinct steps to ensure a clean data flow:

1. **Metrics Implementation:** Pure functions for calculating $Average Precision$ (AP), $Reciprocal Rank$ (RR), and Normalized Discounted Cumulative Gain ($NDCG$).
2. **Data Ingestion (Step 1):** Random generation of queries and document relevance scores.
3. **Processing & Aggregation (Steps 2 & 3):** Iterating through the ingested data to calculate individual query metrics, followed by an aggregation step to compute the overall mean across all queries.

In [2]:
import random
import math

def calculate_rr(relevance_scores):
    for i, rel in enumerate(relevance_scores):
        if rel > 0:
            return 1.0 / (i + 1)
    return 0.0

def calculate_ap(relevance_scores):
    relevant_docs = 0
    precision_sum = 0.0
    total_relevant = sum(1 for x in relevance_scores if x > 0)
    if total_relevant == 0:
        return 0.0
    for i, rel in enumerate(relevance_scores):
        if rel > 0:
            relevant_docs += 1
            precision_sum += relevant_docs / (i + 1)
    return precision_sum / total_relevant

def calculate_ndcg(relevance_scores):
    def get_dcg(scores):
        return sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(scores))
    actual_dcg = get_dcg(relevance_scores)
    ideal_scores = sorted(relevance_scores, reverse=True)
    ideal_dcg = get_dcg(ideal_scores)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg

### Pipeline Class and Execution

The `EvaluationFlow` class manages the state of the data as it moves through the pipeline.
- `step_1_ingestion()` handles input preparation.
- `step_2_processing()` computes the metrics for each individual query.
- `step_3_aggregation()` calculates the mean metrics.
- `execute()` orchestrates the entire flow sequentially.

In [3]:
class EvaluationFlow:
    def __init__(self, num_queries=5, answers_per_query=3):
        self.num_queries = num_queries
        self.answers_per_query = answers_per_query
        self.queries_data = {}
        self.results = {}
        self.averages = {}

    def step_1_ingestion(self):
        possible_indices = [0, 1, 2, 3]
        for q_id in range(1, self.num_queries + 1):
            self.queries_data[f"Query_{q_id}"] = [random.choice(possible_indices) for _ in range(self.answers_per_query)]

    def step_2_processing(self):
        for q_id, scores in self.queries_data.items():
            self.results[q_id] = {
                "AP": calculate_ap(scores),
                "RR": calculate_rr(scores),
                "NDCG": calculate_ndcg(scores)
            }
            print(f"{q_id} | Relevance Scores: {scores}")
            print(f"AP: {self.results[q_id]['AP']:.3f} | RR: {self.results[q_id]['RR']:.3f} | NDCG: {self.results[q_id]['NDCG']:.3f}\n")

    def step_3_aggregation(self):
        print("-" * 30)
        print("AVERAGE METRICS FOR ALL QUERIES")
        print("-" * 30)
        for metric in ["AP", "RR", "NDCG"]:
            total = sum(res[metric] for res in self.results.values())
            self.averages[metric] = total / self.num_queries
            print(f"Mean {metric}: {self.averages[metric]:.3f}")

    def execute(self):
        self.step_1_ingestion()
        self.step_2_processing()
        self.step_3_aggregation()

pipeline = EvaluationFlow(num_queries=5, answers_per_query=3)
pipeline.execute()

Query_1 | Relevance Scores: [0, 0, 2]
AP: 0.333 | RR: 0.333 | NDCG: 0.500

Query_2 | Relevance Scores: [2, 3, 3]
AP: 1.000 | RR: 1.000 | NDCG: 0.845

Query_3 | Relevance Scores: [2, 2, 2]
AP: 1.000 | RR: 1.000 | NDCG: 1.000

Query_4 | Relevance Scores: [1, 1, 3]
AP: 1.000 | RR: 1.000 | NDCG: 0.631

Query_5 | Relevance Scores: [2, 1, 2]
AP: 1.000 | RR: 1.000 | NDCG: 0.951

------------------------------
AVERAGE METRICS FOR ALL QUERIES
------------------------------
Mean AP: 0.867
Mean RR: 0.867
Mean NDCG: 0.786
